In [1]:
import jax.numpy as jnp
from diffrax import diffeqsolve, ODETerm, Dopri5


def f(t, y, args):
    return -y


term = ODETerm(f)
solver = Dopri5()
y0 = jnp.array([2., 3.])
solution = diffeqsolve(term, solver, t0=0, t1=1, dt0=0.1, y0=y0)

In [ ]:
solution.t0

In [ ]:
solution.ys.shape

In [1]:
import flax.linen as nn
from model.actor_critic_rnn import NeuralODE
import jax
import jax.numpy as jnp

rng = jax.random.PRNGKey(0)
coords = jnp.ones((1, 4))

model = NeuralODE(
    encoder=nn.Dense(10),
    derivative_net=nn.Dense(10),
    decoder=nn.Dense(4))
params = jax.jit(model.init)(rng, coords)

In [2]:
@jax.jit
def compute_loss(params, coords, true_coords):
    preds = model.apply(params, coords)
    return jnp.abs(preds - true_coords).sum()


grads = jax.grad(compute_loss)(params, coords, jnp.zeros_like(coords))

In [14]:
import jax
import jax.numpy as jnp
import optax

In [2]:
costs = jnp.array([[[2.1536477 , 2.7017763 , 2.0492606 ],
        [0.8976828 , 2.7900887 , 1.3230644 ],
        [0.8135518 , 1.4669538 , 0.44517508]],

       [[2.5109036 , 2.318946  , 2.1817749 ],
        [2.2890368 , 1.0127853 , 2.1383474 ],
        [2.7151723 , 1.6208652 , 2.499429  ]]]
)

In [3]:
@jax.vmap
def fun(costs):
    agent_idx, landmark_idx = optax.assignment.hungarian_algorithm(costs)
    return agent_idx, landmark_idx

In [4]:
fun(costs)

(Array([[-1, -1, -1],
        [-1, -1, -1]], dtype=int32),
 Array([[0, 1, 2],
        [0, 1, 2]], dtype=int32))

In [2]:
from ortools.graph.python import linear_sum_assignment
import jax.numpy as jnp

In [6]:
assignment = linear_sum_assignment.SimpleLinearSumAssignment()

In [3]:
costs = jnp.array([[2.1536477 , 2.7017763 , 2.0492606 ],
        [0.8976828 , 2.7900887 , 1.3230644 ],
        [0.8135518 , 1.4669538 , 0.44517508]])

In [8]:
end_nodes_unraveled, start_nodes_unraveled = jnp.meshgrid(
    jnp.arange(costs.shape[1]), jnp.arange(costs.shape[0])
)
start_nodes = start_nodes_unraveled.ravel()
end_nodes = end_nodes_unraveled.ravel()
arc_costs = costs.ravel()

In [9]:
assignment.add_arcs_with_cost(start_nodes, end_nodes, arc_costs)

array([0, 1, 2, 3, 4, 5, 6, 7, 8], dtype=int32)

In [10]:
status = assignment.solve()

In [11]:
if status == assignment.OPTIMAL:
    print(f"Total cost = {assignment.optimal_cost()}\n")
    for i in range(0, assignment.num_nodes()):
        print(
            f"Worker {i} assigned to task {assignment.right_mate(i)}."
            + f"  Cost = {assignment.assignment_cost(i)}"
        )
elif status == assignment.INFEASIBLE:
    print("No assignment is possible.")
elif status == assignment.POSSIBLE_OVERFLOW:
    print("Some input costs are too large and may cause an integer overflow.")

Total cost = 2

Worker 0 assigned to task 1.  Cost = 2
Worker 1 assigned to task 0.  Cost = 0
Worker 2 assigned to task 2.  Cost = 0


In [4]:
from scipy.optimize import linear_sum_assignment

In [7]:
type(linear_sum_assignment(costs)[0])

numpy.ndarray

In [15]:
optax.assignment.hungarian_algorithm(costs)

(Array([-1, -1, -1], dtype=int32), Array([0, 1, 2], dtype=int32))